# KSA NFI GeoAI Pipeline V4.2 SCIENTIFIC — **National (entire Kingdom)**

Scientifically revised Colab workflow for the **full KSA NFI** cleansed inventory (all regions), not Asir-only.

### Design principles
1. **National frame first:** all SUs / plots / trees across 13 administrative regions.
2. **Anti-clip:** retain zero-tree SUs in the national universe (no presence bias).
3. **No circular features:** do not use basal area (f(DBH)) as a height predictor.
4. **Honest validation:** random CV vs SU-blocked CV optimism gap.
5. **Sample ≠ design-based:** bootstrap CIs OK; national stock totals **BLOCKED** without design weights.
6. **Field authority:** no remote height overwrite into official carbon.
7. **Asir gates:** optional validation subsection only (430/240/190 GIS package when shapefile present).


## 0. Locate project (Drive **or** auto-clone from GitHub)


In [ ]:
from pathlib import Path
import os, sys, json, warnings, subprocess, shutil
warnings.filterwarnings('ignore')

IN_COLAB = False
try:
    from google.colab import drive
    try:
        drive.mount('/content/drive', force_remount=False)
    except Exception as e:
        print('Drive mount skipped/failed:', e)
    IN_COLAB = True
except Exception:
    pass

REPO_URL = 'https://github.com/gabrielsanya-ops/NFI_KSA_Colab_V4.git'
CLONE_DIR = Path('/content/NFI_KSA_Colab_V4')
PROJECT_ALIASES = ['NFI_KSA_Colab_V4', 'NFI_Asir_Colab_V4']

def looks_like_root(p: Path) -> bool:
    return (p / 'config.yaml').exists() and (p / 'data' / 'cleansed').exists()

CANDIDATES = []
for name in PROJECT_ALIASES:
    CANDIDATES += [
        Path('/content/drive/MyDrive') / name,
        Path('/content/drive/MyDrive/33.Projects') / name,
        Path('/content') / name,
        Path(r'G:/33.Projects') / name,
        Path(r'F:/1.KSA/29.NFI') / name,
    ]
CANDIDATES += [Path.cwd(), CLONE_DIR]

ROOT = next((c.resolve() for c in CANDIDATES if c.exists() and looks_like_root(c)), None)

if ROOT is None and IN_COLAB:
    print('Project not on Drive — cloning from GitHub...')
    if CLONE_DIR.exists():
        shutil.rmtree(CLONE_DIR, ignore_errors=True)
    subprocess.check_call(['git', 'clone', '--depth', '1', REPO_URL, str(CLONE_DIR)])
    ROOT = CLONE_DIR.resolve()

if ROOT is None or not looks_like_root(ROOT):
    raise FileNotFoundError(
        'Could not find NFI package. Either sync folder to Drive MyDrive '
        'or ensure GitHub repo NFI_KSA_Colab_V4 is public, then re-run.'
    )

os.chdir(ROOT)
sys.path.insert(0, str(ROOT / 'src'))
print('IN_COLAB', IN_COLAB)
print('ROOT', ROOT)
print('cleansed files', sorted(p.name for p in (ROOT/'data'/'cleansed').glob('*.xlsx')))


## 1. Install and imports


In [ ]:
%pip install -q numpy pandas openpyxl pyyaml scikit-learn scipy matplotlib seaborn pyshp joblib
import numpy as np, pandas as pd, yaml, matplotlib.pyplot as plt, seaborn as sns
from IPython.display import display
from sklearn.model_selection import GroupKFold, KFold
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

from nfi_v4.biomass import (
    apply_tree_metrics, exclude_by_substrings, shrub_agb_mg, tree_bgb_mg,
    carbon_mg, total_biomass_mg, FORM_FACTOR, WOOD_DENSITY, BEF, ROOT_SHOOT, CARBON_FRACTION,
)
from nfi_v4.gates import anti_clip_su_tree_counts, build_su_plot_lattice, gate1_summary, gate2_summary
from nfi_v4.qc import flag_aberrant_hd, bootstrap_mean_ci
from nfi_v4.sensitivity import one_at_a_time_sensitivity
from nfi_v4.spatial_cv import compare_random_vs_group_cv

sns.set_theme(style='whitegrid'); RS = 42; np.random.seed(RS)

def resolve(p):
    p = Path(p); return p if p.is_absolute() else (ROOT / p).resolve()

def find_col(df, cands):
    m = {c.lower(): c for c in df.columns}
    for c in cands:
        if c.lower() in m: return m[c.lower()]
    for col in df.columns:
        cl = col.lower()
        for c in cands:
            if c.lower() in cl: return col
    return None

def make_plot_uid(df, su_col, plot_col):
    return df[su_col].astype(str).str.strip() + '-' + df[plot_col].astype(str).str.strip()

with open(ROOT / 'config.yaml', encoding='utf-8') as f:
    cfg = yaml.safe_load(f)
for section in ('paths', 'tables'):
    for k, v in list(cfg.get(section, {}).items()):
        cfg[section][k] = str(resolve(v))

OUT = Path(cfg['paths']['output_root'])
for sub in ['tables', 'figures', 'maps', 'reports', 'logs']:
    (OUT / sub).mkdir(parents=True, exist_ok=True)

print('scope', cfg['project'].get('scope'), '| constants FF,WD,BEF,RS,CF =',
      FORM_FACTOR, WOOD_DENSITY, BEF, ROOT_SHOOT, CARBON_FRACTION)


## 2. Ingest cleansed tables (national)


In [ ]:
tables = {k: pd.read_excel(v) for k, v in cfg['tables'].items()}
for k, v in tables.items():
    print(f'{k:12s} n={len(v):6d} cols={v.shape[1]}')

su_all = tables['su'].copy()
plot_all = tables['plot'].copy()
trees_all = tables['trees'].copy()
shrubs_all = tables['shrubs'].copy()

su_id = find_col(su_all, [cfg['columns']['su_id_in_su_master'], 'su_no'])
region_col = find_col(su_all, [cfg['columns']['region_en'], 'su_group_region_e'])
child_su = find_col(plot_all, [cfg['columns']['su_id_in_child'], 'nfi_of_ksa_program_su_su_no', 'su_no'])
tree_su = find_col(trees_all, [cfg['columns']['su_id_in_child'], 'nfi_of_ksa_program_su_su_no', 'su_no'])
plot_id_col = find_col(plot_all, [cfg['columns']['plot_id'], 'plot_id1', 'plot_id'])
tree_plot_col = find_col(trees_all, ['plot_plot_id1', 'plot_id1', 'plot_id'])
dbh_col = find_col(trees_all, [cfg['columns']['dbh'], 'tree_dbh', 'dbh'])
ht_col = find_col(trees_all, [cfg['columns']['height'], 'tree_height', 'height'])
species_col = find_col(trees_all, ['tree_species', 'species', 'scientific', 'sp_name'])
lat_col = find_col(plot_all, [cfg['columns']['lat'], 'plot_info1_plot_coor1_y', 'lat', 'y'])
lon_col = find_col(plot_all, [cfg['columns']['lon'], 'plot_info1_plot_coor1_x', 'lon', 'x'])

print('key cols', dict(su_id=su_id, region=region_col, child_su=child_su, tree_su=tree_su,
                       dbh=dbh_col, ht=ht_col, lat=lat_col, lon=lon_col))
if region_col:
    display(su_all[region_col].value_counts(dropna=False).rename('n_SU').to_frame())


## 3. National anti-clip universe (entire country)

Use **all** master SUs as the analytical frame. Zero-tree SUs are retained (anti-clip).
Asir GIS 430-SU gates are deferred to a later optional validation cell.


In [ ]:
# NATIONAL frame = every SU in master table
national_su_ids = su_all[su_id].astype(str).str.strip().tolist()
ids = set(national_su_ids)

su = su_all.copy()
plot = plot_all[plot_all[child_su].astype(str).str.strip().isin(ids)].copy() if child_su else plot_all.copy()
trees = trees_all[trees_all[tree_su].astype(str).str.strip().isin(ids)].copy() if tree_su else trees_all.copy()

# Attach region to trees via SU master
su_map = su_all[[su_id, region_col]].copy() if region_col else su_all[[su_id]].copy()
su_map[su_id] = su_map[su_id].astype(str).str.strip()
su_map = su_map.rename(columns={su_id: 'su_key'})
if region_col:
    su_map = su_map.rename(columns={region_col: 'region'})

print(f'NATIONAL frame: SUs={len(national_su_ids)} plots={len(plot)} trees={len(trees)}')
print('regions', su_all[region_col].nunique() if region_col else 'n/a')


## 4. Tree QC, exclusions, provisional biomass (national)


In [ ]:
if species_col:
    keep = exclude_by_substrings(trees[species_col], cfg['exclusions']['tree_species_substrings'])
    trees_live = trees.loc[keep].copy()
    n_excl = int((~keep).sum())
else:
    trees_live = trees.copy(); n_excl = 0

valid = trees_live[dbh_col].notna() & trees_live[ht_col].notna()
trees_m = apply_tree_metrics(trees_live.loc[valid].copy(), dbh_col, ht_col)
trees_m['su_key'] = trees_m[tree_su].astype(str).str.strip()
if tree_plot_col is not None:
    trees_m['plot_uid'] = make_plot_uid(trees_m, tree_su, tree_plot_col)
trees_m = trees_m.merge(su_map, on='su_key', how='left')

hd = flag_aberrant_hd(trees_m, dbh_col, ht_col)
trees_m = hd
trees_m['hd_flag'] = trees_m['flag_qc_any'].fillna(False)
trees_fit = trees_m.loc[~trees_m['hd_flag']].copy()

print(f'excluded_species={n_excl} with_metrics={len(trees_m)} hd_flags={int(trees_m.hd_flag.sum())} fit_n={len(trees_fit)}')
display(trees_m[[dbh_col, ht_col, 'basal_area_m2', 'volume_m3', 'agb_mg', 'carbon_mg_provisional']].describe().T)


## 5. National Gate 1 / lattice presence (anti-clip)


In [ ]:
su_tree = anti_clip_su_tree_counts(national_su_ids, trees_m, 'su_key')
su_tree = su_tree.merge(su_map, on='su_key', how='left')

national_g1 = {
    'scope': 'national_KSA',
    'observed_total_sus': int(len(su_tree)),
    'observed_sus_with_trees': int(su_tree['has_trees'].sum()),
    'observed_sus_no_trees': int((~su_tree['has_trees']).sum()),
    'n_plots_in_scope': int(len(plot)),
    'n_trees_with_metrics': int(len(trees_m)),
    'note': 'SAMPLE_FRAME_ANTI_CLIP_NOT_DESIGN_BASED',
}
print(json.dumps(national_g1, indent=2))

# Regional breakdown (anti-clip)
if 'region' in su_tree.columns:
    reg = (su_tree.groupby('region', dropna=False)
           .agg(n_su=('su_key', 'size'), n_su_with_trees=('has_trees', 'sum'))
           .assign(n_su_zero_trees=lambda d: d['n_su'] - d['n_su_with_trees'])
           .sort_values('n_su', ascending=False))
    display(reg)
    reg.to_csv(OUT / 'tables' / 'national_su_tree_presence_by_region.csv')

lattice = build_su_plot_lattice(national_su_ids, cfg['project']['plots_per_su'], 'su_key')
g2_nat = gate2_summary(lattice, trees_m, 'su_key', plot_uid_col='plot_uid', expected=None)
print('national lattice plots', g2_nat['summary'])

su_tree.to_csv(OUT / 'tables' / 'national_su_anti_clip_tree_counts.csv', index=False)
pd.DataFrame([national_g1]).to_csv(OUT / 'tables' / 'national_gate1_summary.csv', index=False)
pd.DataFrame([g2_nat['summary']]).to_csv(OUT / 'tables' / 'national_gate2_summary.csv', index=False)


## 6. Optional Asir validation gates (GIS package)

Runs only if `Asir_SU_Final.shp` is present. Does **not** restrict the national analysis above.


In [ ]:
asir_gate_note = 'Asir shapefile not found — national analysis unaffected'
shp = Path(cfg['paths']['asir_su'])
if shp.exists():
    import shapefile
    r = shapefile.Reader(str(shp))
    fields = [f[0] for f in r.fields[1:]]
    su_field = next((n for n in fields if n.upper() in ('SU', 'SU_NO', 'SU_ID')), fields[0])
    asir_su_ids = [str(rec[fields.index(su_field)]).strip() for rec in r.records()]
    trees_asir = trees_m[trees_m['su_key'].isin(set(asir_su_ids))].copy()
    su_tree_asir = anti_clip_su_tree_counts(asir_su_ids, trees_asir, 'su_key')
    g1_asir = gate1_summary(su_tree_asir, cfg['aseer_gate_expected'])
    g2_asir = gate2_summary(
        build_su_plot_lattice(asir_su_ids, 5, 'su_key'), trees_asir, 'su_key',
        plot_uid_col='plot_uid', expected=cfg['aseer_gate_expected'])
    asir_gate_note = 'Asir validation gates computed against GIS frame'
    print(asir_gate_note)
    print('Gate1', g1_asir)
    print('Gate2', g2_asir['summary'])
    pd.DataFrame([g1_asir]).to_csv(OUT / 'tables' / 'asir_gate1_summary.csv', index=False)
    pd.DataFrame([g2_asir['summary']]).to_csv(OUT / 'tables' / 'asir_gate2_summary.csv', index=False)
else:
    print(asir_gate_note)


## 7. Sample biometrics + provisional carbon (national; not design-based)


In [ ]:
uni_rows = []
for col in [dbh_col, ht_col, 'basal_area_m2', 'volume_m3', 'agb_mg', 'carbon_mg_provisional']:
    s = trees_m[col].dropna()
    ci = bootstrap_mean_ci(s.to_numpy(), n_boot=1000, random_state=RS)
    uni_rows.append({
        'variable': col, 'n': int(len(s)),
        'mean': float(s.mean()), 'median': float(s.median()),
        'sd': float(s.std(ddof=1)) if len(s) > 1 else np.nan,
        'boot_mean': ci['mean'], 'boot_lo': ci['ci_low'], 'boot_hi': ci['ci_high'],
        'note': 'SAMPLE_STATISTICS_NOT_DESIGN_BASED',
    })
uni = pd.DataFrame(uni_rows)
display(uni)
uni.to_csv(OUT / 'tables' / 'national_univariate_tree_biometrics.csv', index=False)

if 'region' in trees_m.columns:
    by_reg = (trees_m.groupby('region', dropna=False)
              .agg(n_trees=(dbh_col, 'size'),
                   mean_dbh=(dbh_col, 'mean'),
                   mean_ht=(ht_col, 'mean'),
                   mean_C=('carbon_mg_provisional', 'mean'))
              .sort_values('n_trees', ascending=False))
    display(by_reg)
    by_reg.to_csv(OUT / 'tables' / 'national_tree_metrics_by_region.csv')

print('POLICY: national stock totals BLOCKED without design weights / inclusion probabilities.')


## 8. Shrubs (national provisional)


In [ ]:
shrubs = shrubs_all.copy()
shrub_note = 'shrubs not processed'
cd1 = find_col(shrubs, ['crown_diam1', 'crown_diameterl1', 'crown_diameter_1', 'cd1'])
cd2 = find_col(shrubs, ['crown_diam2', 'crown_diameterl2', 'crown_diameter_2', 'cd2'])
sh_h = find_col(shrubs, ['sh_height', 'shrub_height', 'height', 'height_m'])
sh_su = find_col(shrubs, [cfg['columns']['su_id_in_child'], 'nfi_of_ksa_program_su_su_no', 'su_no'])
sp = find_col(shrubs, ['species', 'shrub_species', 'sp_name'])
sh = shrubs.copy()
if sh_su is not None:
    sh = sh[sh[sh_su].astype(str).str.strip().isin(ids)]
if sp is not None:
    sh = sh.loc[exclude_by_substrings(sh[sp], cfg['exclusions']['shrub_exclude_substrings'])]
if cd1 and cd2 and sh_h and len(sh):
    sh = sh.copy()
    sh['mean_CD_m'] = (pd.to_numeric(sh[cd1], errors='coerce') + pd.to_numeric(sh[cd2], errors='coerce')) / 2.0
    h_m = pd.to_numeric(sh[sh_h], errors='coerce')
    sh['height_cm'] = np.where(h_m.median(skipna=True) < 20, h_m * 100.0, h_m)
    ok = sh['mean_CD_m'].notna() & sh['height_cm'].notna() & (sh['mean_CD_m'] > 0) & (sh['height_cm'] > 0)
    sh_ok = sh.loc[ok].copy()
    sh_ok['agb_mg'] = shrub_agb_mg(sh_ok['mean_CD_m'], sh_ok['height_cm'])
    sh_ok['bgb_mg'] = tree_bgb_mg(sh_ok['agb_mg'])
    sh_ok['total_biomass_mg'] = total_biomass_mg(sh_ok['agb_mg'], sh_ok['bgb_mg'])
    sh_ok['carbon_mg_provisional'] = carbon_mg(sh_ok['total_biomass_mg'])
    sh_ok.to_csv(OUT / 'tables' / 'national_shrubs_with_provisional_biomass.csv', index=False)
    shrub_note = f'shrubs processed n={len(sh_ok)}'
else:
    shrub_note = f'shrub columns cd1={cd1} cd2={cd2} h={sh_h}; insufficient'
print(shrub_note)


## 9. Height~DBH models + random vs SU-blocked CV (national)


In [ ]:
X = trees_fit[[dbh_col]].to_numpy()
y = trees_fit[ht_col].to_numpy()
groups = trees_fit['su_key'].astype(str).to_numpy()

ols = LinearRegression().fit(X, y)
yhat = ols.predict(X)
print('OLS full-sample R2', float(r2_score(y, yhat)), 'RMSE', float(np.sqrt(mean_squared_error(y, yhat))))

fold_df, summary_cv, gap = compare_random_vs_group_cv(
    RandomForestRegressor(n_estimators=200, random_state=RS, n_jobs=-1, min_samples_leaf=5),
    X, y, groups, n_splits=5, random_state=RS,
)
display(summary_cv)
print('optimism gap', gap)
fold_df.to_csv(OUT/'tables'/'national_cv_folds.csv', index=False)
summary_cv.to_csv(OUT/'tables'/'national_cv_summary.csv', index=False)
pd.DataFrame([gap]).to_csv(OUT/'tables'/'national_cv_optimism_gap.csv', index=False)

med_dbh = float(np.nanmedian(trees_fit[dbh_col])); med_ht = float(np.nanmedian(trees_fit[ht_col]))
sens = one_at_a_time_sensitivity(med_dbh, med_ht)
display(sens)
sens.to_csv(OUT/'tables'/'national_factor_sensitivity.csv', index=False)


## 10. Figures and national map


In [ ]:
fig, ax = plt.subplots(figsize=(7, 5), dpi=120)
ax.hist(trees_m[dbh_col].dropna(), bins=40, color='#2f4f4f', edgecolor='white')
ax.set_xlabel('DBH (cm)'); ax.set_ylabel('Count')
ax.set_title('Fig 1. National KSA tree DBH (sample)')
fig.tight_layout(); fig.savefig(OUT/'figures'/'fig01_national_dbh_hist.png', dpi=200); plt.show()

fig, ax = plt.subplots(figsize=(7, 5), dpi=120)
ax.scatter(trees_fit[dbh_col], trees_fit[ht_col], s=6, alpha=0.25, c='#1f4e79')
xs = np.linspace(trees_fit[dbh_col].min(), trees_fit[dbh_col].max(), 100).reshape(-1, 1)
ax.plot(xs, ols.predict(xs), color='#c0392b', lw=2, label='OLS')
ax.set_xlabel('DBH (cm)'); ax.set_ylabel('Height (m)')
ax.set_title('Fig 2. National height vs DBH (HD-flagged removed)'); ax.legend()
fig.tight_layout(); fig.savefig(OUT/'figures'/'fig02_national_height_dbh.png', dpi=200); plt.show()

fig, ax = plt.subplots(figsize=(8, 4), dpi=120)
ax.bar(['SUs with trees', 'SUs zero trees retained'],
       [national_g1['observed_sus_with_trees'], national_g1['observed_sus_no_trees']],
       color=['#2e7d32', '#9e9e9e'])
ax.set_title('Fig 3. National anti-clip Gate 1'); ax.set_ylabel('Count')
fig.tight_layout(); fig.savefig(OUT/'figures'/'fig03_national_gate1.png', dpi=200); plt.show()

if lat_col and lon_col:
    coords = plot[[child_su, lat_col, lon_col]].copy() if child_su else plot[[lat_col, lon_col]].copy()
    coords[lat_col] = pd.to_numeric(coords[lat_col], errors='coerce')
    coords[lon_col] = pd.to_numeric(coords[lon_col], errors='coerce')
    coords = coords.dropna(subset=[lat_col, lon_col])
    if child_su:
        coords['su_key'] = coords[child_su].astype(str).str.strip()
        coords = coords.merge(su_tree[['su_key', 'has_trees']], on='su_key', how='left')
    fig, ax = plt.subplots(figsize=(9, 8), dpi=120)
    if 'has_trees' in coords.columns:
        for flag, color, lab in [(True, '#0b3d0b', 'SU has trees'), (False, '#bdbdbd', 'SU zero trees')]:
            sub = coords[coords['has_trees'] == flag]
            ax.scatter(sub[lon_col], sub[lat_col], s=4, c=color, alpha=0.45, label=lab)
        ax.legend(markerscale=3)
    else:
        ax.scatter(coords[lon_col], coords[lat_col], s=4, c='#0b3d0b', alpha=0.4)
    ax.set_xlabel('Longitude'); ax.set_ylabel('Latitude')
    ax.set_title('Map 1. National KSA NFI plot locations\nCRS: WGS84 assumed')
    ax.grid(True, alpha=0.3)
    fig.tight_layout(); fig.savefig(OUT/'maps'/'map01_national_plot_locations.png', dpi=200); plt.show()
    coords.to_csv(OUT/'tables'/'national_plot_coordinates.csv', index=False)


## 11. Export national tables + run summary


In [ ]:
trees_m.to_csv(OUT/'tables'/'national_trees_with_qc_biomass.csv', index=False)
trees_fit.to_csv(OUT/'tables'/'national_trees_fit_allometry.csv', index=False)

summary = {
    'scope': 'national_KSA',
    'n_su_frame': national_g1['observed_total_sus'],
    'n_su_with_trees': national_g1['observed_sus_with_trees'],
    'n_plots': int(len(plot)),
    'n_trees_metrics': int(len(trees_m)),
    'n_fit': int(len(trees_fit)),
    'national_g1': national_g1,
    'national_g2': g2_nat['summary'],
    'asir_gate_note': asir_gate_note,
    'shrub_note': shrub_note,
    'policy': cfg['policy'],
    'ROOT': str(ROOT),
}
(OUT/'logs'/'national_run_summary.json').write_text(json.dumps(summary, indent=2, default=str), encoding='utf-8')
print(json.dumps(summary, indent=2, default=str))
print('NATIONAL V4.2 pipeline complete.')
